# Fase CRISP-DM: Evaluación de Modelos, Diagnóstico y Puesta en Servicio
**Plataforma**: AgroData Intelligence Platform (AgroStatsApp)  
**Estándares**: SWEBOK, ISO/IEC 25010, Explicabilidad del Dato, Servicio de Inferencia  
**Propósito**: Deserializar el modelo ganador, computar métricas de precisión y sesgo (MAE, RMSE, WAPE, R²), evaluar la aleatoriedad de los residuos (Durbin-Watson), visualizar la importancia de variables y simular el servicio productivo de inferencia.

In [ ]:
# 1. Configuración de Entorno Resiliente (Google Colab / VS Code / Jupyter Local)
import os
import sys
import subprocess
from pathlib import Path

def setup_environment():
    # A. Detección y preparación automática para Google Colab
    if 'google.colab' in sys.modules or Path('/content').exists():
        print('[INFO] Entorno detectado: Google Colab.')
        repo_dir = Path('/content/Statsfirm')
        if not repo_dir.exists():
            print('[INFO] Clonando repositorio oficial Statsfirm en Colab...')
            subprocess.run(['git', 'clone', 'https://github.com/adansanchezc1-spec/Statsfirm.git', '/content/Statsfirm'], check=True)
        else:
            print('[INFO] Actualizando repositorio en Colab...')
            subprocess.run(['git', '-C', '/content/Statsfirm', 'pull'], check=False)
        
        app_dir = repo_dir / 'AgroStats AndTech' / 'AgroStatsApp'
        if app_dir.exists():
            os.chdir(str(app_dir))
            src_dir = app_dir / 'src'
            if str(src_dir) not in sys.path:
                sys.path.insert(0, str(src_dir))
            print(f'[OK] Directorio de trabajo establecido en: {app_dir}')
            print(f'[OK] Carpeta src agregada a sys.path: {src_dir}')
            return

    # B. Detección dinámica en Entorno Local (Windows / Linux / WSL / VS Code)
    candidates = [
        Path.cwd() / 'src',
        Path.cwd() / 'AgroStats AndTech' / 'AgroStatsApp' / 'src',
        Path.cwd().parent / 'src',
        Path.cwd().parent.parent / 'src',
        Path.cwd().parent.parent / 'AgroStats AndTech' / 'AgroStatsApp' / 'src',
        Path(r'c:\Users\ADAN\OneDrive\Documentos\Statsfirm\AgroStats AndTech\AgroStatsApp\src'),
        Path('/mnt/c/Users/ADAN/OneDrive/Documentos/Statsfirm/AgroStats AndTech/AgroStatsApp/src'),
    ]
    
    curr = Path.cwd().resolve()
    for _ in range(6):
        target = curr / 'AgroStats AndTech' / 'AgroStatsApp' / 'src'
        if target.exists() and (target / 'notebook_code').is_dir():
            candidates.insert(0, target)
            break
        curr = curr.parent

    for c in candidates:
        if c.exists() and (c / 'notebook_code').is_dir():
            resolved = str(c.resolve())
            if resolved not in sys.path:
                sys.path.insert(0, resolved)
            print(f'[OK] Módulo src localizado localmente en: {resolved}')
            return

setup_environment()

# Importación del evaluador y servicio de inferencia
from notebook_code import ModelEvaluator, InferenceService, ModelTrainer, load_all_raw_datasets, FeatureEngineer
import pandas as pd
from pathlib import Path

print('[OK] Módulos de Evaluación e Inferencia importados exitosamente.')


In [ ]:
# 2. Carga del Modelo Persistido y Conjunto de Prueba
model_path = Path('CRISPDM/data/MODEL/best_agro_model.joblib')
feat_path = Path('CRISPDM/data/FEATURES/master_feature_matrix.csv')

df = pd.read_csv(feat_path)
X_train, y_train, X_test, y_test, train_dates, test_dates = ModelTrainer.prepare_train_test_split(
    df,
    target_col='valorobservado',
    date_col='fechaobservacion',
    test_size=0.2
)

pipeline, metadata = ModelEvaluator.load_model(model_path)
print(f'Modelo cargado: {metadata.get("estimator_class")} ({metadata.get("features_count")} variables)')


In [ ]:
# 3. Evaluación Exhaustiva de Métricas (MAE, RMSE, MAPE, WAPE, R²)
test_preds = pipeline.predict(X_test)
metrics = ModelEvaluator.evaluate_comprehensive_metrics(y_test, test_preds)
print('=== MÉTRICAS DE BONDAD DE AJUSTE Y PRECISIÓN ===')
for k, v in metrics.items():
    print(f'• {k:15}: {v}')


In [ ]:
# 4. Diagnóstico Estadístico de Residuos (Test Durbin-Watson y Sesgo)
res_diag = ModelEvaluator.residual_diagnostics(y_test, test_preds)
print('=== DIAGNÓSTICO DE RESIDUOS (e = y - y_pred) ===')
for k, v in res_diag.items():
    print(f'• {k:22}: {v}')


In [ ]:
# 5. Gráfico Diagnóstico: Pronóstico vs Real con Banda de Error
fig_forecast = ModelEvaluator.plot_forecast_vs_actual(
    dates=test_dates,
    y_true=y_test,
    y_pred=test_preds,
    target_name='Precipitación / Variable Agro'
)
if fig_forecast:
    fig_forecast.show()


In [ ]:
# 6. Explicabilidad: Importancia de Variables Predictoras
fig_imp = ModelEvaluator.plot_feature_importance(
    pipeline=pipeline,
    feature_names=metadata.get('features', list(X_test.columns)),
    top_n=12
)
if fig_imp:
    fig_imp.show()


In [ ]:
# 7. Distribución de Residuos frente a Normalidad Teórica
fig_res = ModelEvaluator.plot_residual_distribution(y_test, test_preds)
if fig_res:
    fig_res.show()


In [ ]:
# 8. Simulación del Servicio de Inferencia en Producción
service = InferenceService(model_path)
sample_new_data = X_test.head(5)
scored_df = service.predict(sample_new_data)
print('=== PREDICCIONES GENERADAS POR InferenceService ===')
display(scored_df[['prediccion_modelo', 'modelo_version'] + list(sample_new_data.columns[:3])])
